# Sprint 4 — RAG com página, seção e estado operacional

**Challenge FIAP — Forzy | Processamento de Linguagem Natural**

Este notebook compara o preset legado com uma recuperação em português que preserva a origem de cada trecho e usa alertas ativos para orientar a busca.

| Seção | Conteúdo |
|---|---|
| 0 | Objetivo e roteiro |
| 1 | Setup local e chave do OpenRouter |
| 2 | Extração por página, chunking e comparação dos presets |
| 3 | Embeddings, reranker e indexação |
| 4 | Estado operacional a partir das leituras reais |
| 5 | Busca com e sem estado |
| 6 | Conclusões e limites |
| 7 | Assistente de troubleshooting |
| 8 | Smoke do assistente |


## 1. Setup

Execute a partir da raiz do repositório. O bloco localiza a raiz sem clonar o projeto nem instalar pacotes; as dependências ficam em `requirements.txt`. A chave é lida do `.env` da raiz e nunca exibida.


In [ ]:
from pathlib import Path
import sys

candidatos_raiz = (Path.cwd().resolve(), *Path.cwd().resolve().parents)
ROOT = next((candidato for candidato in candidatos_raiz if (candidato / "src" / "pln" / "config.py").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Abra o notebook dentro do repositório que contém src/pln/config.py.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.pln.config import NOVO, LEGADO, PDF_CATALOGO_FILE
from src.pln.llm import load_openrouter_api_key

try:
    chave_openrouter_disponivel = bool(load_openrouter_api_key(ROOT / ".env"))
except RuntimeError:
    chave_openrouter_disponivel = False

print(f"Raiz do repositório: {ROOT}")
print(f"PDF do catálogo encontrado: {PDF_CATALOGO_FILE.is_file()}")
print(f"Chave OpenRouter disponível: {chave_openrouter_disponivel}")


## 2. Extração e chunking

A extração remove os elementos repetidos do cabeçalho/rodapé, mantém o número impresso, identifica seções e converte tabelas em linhas rotuladas. O preset `LEGADO` mantém o agrupamento por parágrafo sem metadados; `NOVO` mantém página e seção.


In [ ]:
from src.pln.config import LEGADO, NOVO, PDF_CATALOGO_FILE
from src.pln.rag import chunk_legado, chunk_novo, extrair_paginas

paginas = extrair_paginas(PDF_CATALOGO_FILE)
chunks_legado = chunk_legado("\n\n".join(pagina.texto_original for pagina in paginas), LEGADO)
chunks_novos = chunk_novo(paginas, NOVO)

print(f"Páginas extraídas: {len(paginas)}")
print(f"Chunks LEGADO: {len(chunks_legado)} | limite: {LEGADO.chunk_max} caracteres")
print(f"Chunks NOVO: {len(chunks_novos)} | limite: {NOVO.chunk_max} caracteres")
print(f"Chunks NOVO com página e seção: {sum(chunk.pagina is not None and bool(chunk.secao) for chunk in chunks_novos)}")
for chunk in chunks_novos[:3]:
    print(f"\nPágina {chunk.pagina} — seção {chunk.secao} — {chunk.secao_titulo}\n{chunk.texto}")


## 3. Indexação

O índice usa ChromaDB efêmero com distância cosseno. O preset `NOVO` aplica embeddings E5 multilíngues e reranking multilíngue. O tempo abaixo inclui o carregamento dos modelos na primeira execução e a criação do índice.


In [ ]:
from time import perf_counter

from src.pln.rag import criar_retriever

inicio_indexacao = perf_counter()
retriever = criar_retriever(chunks_novos, NOVO)
tempo_indexacao = perf_counter() - inicio_indexacao
print(f"Embedding: {NOVO.modelo_embedding}")
print(f"Reranker: {NOVO.modelo_reranker}")
print(f"Passagens indexadas: {len(chunks_novos)}")
print(f"Tempo de carregamento e indexação: {tempo_indexacao:.1f} s")


## 4. Estado operacional

Os episódios são derivados das leituras reais de sensores. A primeira amostra usa o início de um alerta crítico; a segunda usa uma leitura real fora de qualquer janela de alerta.


In [ ]:
from src.pln.alertas import carregar_leituras, detectar_episodios
from src.pln.estado import construir_estado_operacional

leituras = carregar_leituras()
alertas = detectar_episodios(leituras)
alerta_critico = next(alerta for alerta in alertas if alerta.severidade == "critico")
instante_critico = alerta_critico.inicio
instantes_leitura = sorted((leitura["timestamp"] for leitura in leituras), reverse=True)
instante_normal = next(
    instante
    for instante in instantes_leitura
    if not any(alerta.inicio <= instante <= alerta.fim for alerta in alertas)
)
estado_critico = construir_estado_operacional(alertas, instante_critico)
estado_normal = construir_estado_operacional(alertas, instante_normal)

print(f"Episódios derivados: {len(alertas)}")
print(f"Instante crítico: {estado_critico.texto()}")
print(f"Instante normal: {estado_normal.texto()}")


## 5. Retriever com e sem estado

A mesma pergunta percorre o retriever em duas condições. Sem estado, não há expansão por termos operacionais nem bônus de seção. Com estado, a busca inclui os termos do domínio ativo e pode receber o bônus definido no preset.


In [ ]:
pergunta = "Quais verificações devo fazer para entender o aquecimento do motor?"

resultados_sem_estado = retriever.buscar(pergunta, estado=None)
resultados_com_estado = retriever.buscar(pergunta, estado=estado_critico)

def mostrar_resultados(rotulo, resultados):
    print(f"\n{rotulo}")
    for posicao, resultado in enumerate(resultados, start=1):
        print(
            f"{posicao}. p. {resultado['pagina']} — seção {resultado['secao']} — "
            f"{resultado['secao_titulo']} | denso={resultado['score_denso']:.3f} "
            f"rerank={resultado['score_rerank']:.3f} final={resultado['score_final']:.3f}"
        )
        print(resultado["texto"][:320])

mostrar_resultados("Sem estado", resultados_sem_estado)
mostrar_resultados("Com estado crítico", resultados_com_estado)


## 6. Conclusão

O preset `NOVO` mantém página e seção nos chunks, serializa tabelas em linhas legíveis e expande a busca com termos associados aos alertas ativos. `LEGADO` permanece disponível para a comparação planejada.

A seção 5 permite observar a mudança de consulta e de ordenação com o mesmo enunciado. O resultado depende dos modelos multilíngues configurados e do catálogo W22; não deve ser interpretado como avaliação de qualidade sem o conjunto de perguntas e métricas da próxima etapa.

**Execução local:** rode as células em ordem para baixar/carregar os modelos e verificar os resultados reais. Não salve outputs ou chaves no notebook.


## 7. Assistente de troubleshooting

O assistente usa o estado operacional como dado confiável e os chunks recuperados como única fonte técnica. O score bruto do reranker determina abstenção e confiança; página e seção dos chunks validam as citações. A memória mantém até `MEMORIA_TURNOS` turnos e amplia consultas curtas ou anafóricas com a pergunta anterior.

### Inspeção offline e limiar provisório

Inspecionei 10 consultas dentro do escopo — incluindo duas buscas já compostas com a pergunta anterior — e 6 consultas fora dele. Os valores abaixo são `score_rerank_bruto` do primeiro chunk retornado pelo retriever, usando o mesmo estado crítico passado ao smoke. A regra provisória é somar a margem mínima ao maior score fora do escopo e arredondar o resultado para cima ao décimo: `ceil((maior_score_fora + margem) * 10) / 10`.

O maior score fora do escopo foi -3.027231. Com margem mínima de 0.1, o valor intermediário é -2.927231 e o arredondamento para cima resulta em `LIMIAR_ABSTENCAO = -2.9` (margem efetiva de 0.127231 sobre o maior score externo). Não há limiar que separe as duas populações nesta amostra: o menor score dentro do escopo (-4.751304, seguimento de ventilação) é menor que o maior score fora dele (-3.027231). Com o limiar escolhido, as seis consultas externas abstêm e a pergunta de ventilação e seu seguimento permanecem como casos abertos para a plan 02.03. Ambos continuam registrados na tabela.

| Escopo | Consulta usada na medição | Melhor score bruto | Página/seção do primeiro chunk |
|---|---|---:|---|
| Dentro | Qual é o grau de proteção do motor WEG W22? | 9.925054 | 18 / 7 |
| Dentro | Qual é o grau de proteção do motor WEG W22? E o que essa proteção significa? | 8.729171 | 18 / 7 |
| Dentro | Qual é a classe de isolamento do motor WEG W22? | 6.704764 | 20 / 11.1 |
| Dentro | Quais são as características do sistema de ventilação? | -4.239105 | 21 / 12.1 |
| Dentro | Quais são as características do sistema de ventilação? E quais são as características desse sistema? | -4.751304 | 21 / 12.1 |
| Dentro | Quais dados técnicos aparecem na placa de identificação do motor WEG W22? | 9.336778 | 12 / 3.11 |
| Dentro | Qual é a faixa de temperatura ambiente permitida para o motor WEG W22? | 7.954395 | 19 / 9 |
| Dentro | Qual altitude de instalação é permitida para o motor WEG W22? | 6.590040 | 20 / 11.1 |
| Dentro | Como deve ser feita a lubrificação dos rolamentos do motor WEG W22? | 4.264139 | 14 / 5.1 |
| Dentro | Quais são os limites de vibração permitidos para o motor WEG W22? | 7.368129 | 14 / 5.2 |
| Fora | Como assar pão de queijo? | -6.147628 | 20 / 11.1 |
| Fora | Qual é a população atual do Brasil? | -3.027231 | 38 / 16 |
| Fora | Como configurar uma impressora a laser? | -6.134845 | 33 / 15 |
| Fora | Qual é a capital da França? | -3.982897 | 38 / 16 |
| Fora | Como preparar um bolo de chocolate? | -6.812472 | 38 / 16 |
| Fora | Como configurar um roteador Wi-Fi doméstico? | -5.741543 | 33 / 15 |


In [ ]:
from IPython.display import Markdown, display
from src.pln.config import (
    FAIXAS_CONFIANCA,
    LIMIAR_ABSTENCAO,
    MARGEM_ABSTENCAO,
    LIMIAR_PALAVRAS_CONSULTA,
    MEMORIA_TURNOS,
    MODELO_GERADOR,
    PROMPT_SISTEMA_ASSISTENTE,
)

display(Markdown(f"```text\n{PROMPT_SISTEMA_ASSISTENTE}\n```"))
display(Markdown(
    "|Parâmetro|Valor|\n|---|---|\n"
    f"|Modelo gerador|`{MODELO_GERADOR}`|\n"
    f"|Memória em turnos|`{MEMORIA_TURNOS}`|\n"
    f"|Limiar de consulta curta|`{LIMIAR_PALAVRAS_CONSULTA}`|\n"
    f"|Limiar provisório de abstenção|`{LIMIAR_ABSTENCAO}`|\n"
    f"|Margem mínima do limiar|`{MARGEM_ABSTENCAO}`|\n"
    f"|Faixas de confiança|`{FAIXAS_CONFIANCA}`|"
))


## 8. Smoke do assistente

Execute as três perguntas dentro do escopo, os dois seguimentos de memória e as três perguntas fora do escopo. Cada resultado mostra a consulta usada, fontes, confiança, abstenção e modelo. Para respostas não abstidas, o smoke também imprime o texto completo, incluindo citações e a linha de confiança. Esta seção faz chamadas ao gerador e depende da chave local do OpenRouter.


In [ ]:
from src.pln.assistente import Assistente

perguntas_smoke = (
    "Qual é o grau de proteção do motor WEG W22?",
    "E o que essa proteção significa?",
    "Qual é a classe de isolamento do motor WEG W22?",
    "Quais são as características do sistema de ventilação?",
    "E quais são as características desse sistema?",
    "Como assar pão de queijo?",
    "Qual é a população atual do Brasil?",
    "Como configurar uma impressora a laser?",
)
assistente_demo = Assistente(retriever)
respostas_smoke = []
for pergunta_smoke in perguntas_smoke:
    resposta_smoke = assistente_demo.responder(pergunta_smoke, estado=estado_critico)
    respostas_smoke.append(resposta_smoke)
    print(
        {
            "pergunta": pergunta_smoke,
            "consulta_usada": resposta_smoke.consulta_usada,
            "fontes": resposta_smoke.fontes,
            "confianca": resposta_smoke.confianca,
            "abstencao": resposta_smoke.abstencao,
            "modelo": resposta_smoke.modelo,
        }
    )
    if not resposta_smoke.abstencao:
        print(resposta_smoke.texto)
